In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Thu Oct 30 16:41:57 2025

@author: kabroc001
"""


##TODO: add datetime to logged data!! for comparison to Sensordata!

## original script from Nora

########################################

# this is the code which worked last time

import time
import serial
import sched
#from pymeasure.log import console_log
#from pymeasure.display import Plotter
#from pymeasure.experiment import Procedure, Worker
#from pymeasure.experiment import IntegerParameter
import queue
import select
import sys
import os
from datetime import datetime

## libraries for live plotting:
#import dash
#from dash.dependencies import Output, Input
#from dash import dcc
#from dash import html
#import plotly
#import random
#import plotly.graph_objs as go
#from collections import deque

#from threading import Thread

#%%
# Micro K anschalten, nicht auf Resume drücken

### param:
# Get the absolute path of the script
script_dir = os.path.dirname(os.path.abspath("__file__"))
print(script_dir)


# Get the variables from the param txt file:
commands_lists = []
with open(os.path.join(script_dir, "paramMicroK.txt"), "r") as file: #sys.argv[1]
        lines = file.readlines()
        for line in lines:
            if not line.startswith("#"): # Ignore lines starting with '#'
                commands = line.strip().split(';')
                commands_lists.append(commands)
f_name, Flukefreq, Fluketemps, Bridgecommands = commands_lists[:4]  # nur die ersten 4 Zeilen; Rest = LoggerSensor-Settings

print(Bridgecommands)

start_time = time.time()
# exp_len = (len(Fluketemps)*int(Flukefreq[0])*60)+5*60

# open serial port:
port = "/dev/cu.usbserial-A922BJHF" #"/dev/cu.usbserial-A9PH3CPR"  #"/dev/cu.usbserial-A9LMGQ4E"  "/dev/cu.usbserial-FTG9EKPY" # anderes Kabel (usbc port) #"/dev/cu.usbserial-A907B0QG"
baudrate = 9600
bytesize = serial.EIGHTBITS  # 8 bits per byte
parity = serial.PARITY_NONE  # No parity
stopbits = serial.STOPBITS_ONE  # 1 stop bit
timeout =  2 #1 (default)

# Open the serial port
ser = serial.Serial(port, baudrate, bytesize, parity, stopbits, timeout)

if ser.isOpen():
    print("Serial port is open")
else:
    print("Failed to open serial port")

#%%
#total_timeout = 360

#timeout_message = "MicroK is not responding in set timeout."

#%%
"""
# this is my test area
print(ser.name)

test_command = "*IDN? \r\n"

#ser.query("*IDN?") # query ist kein Befehl in serial


ser.write(test_command.encode("ascii"))
test_readout = ser.readline(128).decode("utf-8").strip()
test_readout = ser.read(8).decode("utf-8").strip()

#test_readout = ser.read_until("\r\n").decode("utf-8").strip()
#print(test_readout)
"""
#%%

##
commands = Bridgecommands
    
file_name = time.strftime("%Y%m%d-%H%M%S")+ f_name[0] + "MicroK" + ".txt"

bridgecommands1 = ["MEAS:RAT" + str(commands[1]) + ":REF" + str(commands[0]) + "? 100,0.56 \r\n", 
                   "MEAS:RAT" + str(commands[1]) + ":REF" + str(commands[0]) + "? 100,0.56 \r\n"]#MEAS:RAT1:REF204?\r\n”; “MEAS:RAT2:REF204?\r\n

print(bridgecommands1)


if len(commands) == 3:
    # if there are two SPRTs:
    bridgecommands2 = ["MEAS:RAT" + str(commands[2]) + ":REF" + str(commands[0]) + "? 100,0.56 \r\n", 
                       "MEAS:RAT" + str(commands[2]) + ":REF" + str(commands[0]) + "? 100,0.56 \r\n"]#MEAS:RAT1:REF204?\r\n”; “MEAS:RAT2:REF204?\r\n

    print(bridgecommands2)

#ser.write(checkc.encode("ascii"))
#received_data = str()
#%%
start_getdata = time.time()
while(True):
#while((time.time()-start_getdata )<total_timeout):    
    # depending on wheather there are one or two SPRT:
    f = 1
    while f < 101:
        f = f+1
        command = bridgecommands1[0]

        #send command to bridge: 
        ser.write(command.encode("ascii"))

        data = str()
        while(not data):
            data = ser.readline().decode("utf-8").strip()

        #print(data)

        ## append serial data:
        prog_datetime = datetime.now()
        prog_time = time.time() - start_time
        prog_time = prog_time/60
        
        #print(str(prog_time) + ";" + str(data) + ";1mA" + ";Channel" + str(commands[1]) + ";" + str(f) + "\n") 

        # save data to .txt here:
        with open(file_name, "a") as data_file:
            data_file.write(str(prog_time) + ";" + str(prog_datetime) + ";" + str(data) + ";0.56mA" + ";Channel" + str(commands[1]) + ";" + str(f) + "\n") 


    j = 1
    while j < 101:
        j = j+1

        command = bridgecommands1[1]
        #print(command)

        #send command to bridge: 
        ser.write(command.encode("ascii"))

        data = str()
        while(not data):
            data = ser.readline().decode("utf-8").strip()

        #print(data)


        ## append serial data:
        prog_datetime = datetime.now()
        prog_time = time.time() - start_time
        prog_time = prog_time/60


        # save data to .txt here:
        with open(file_name, "a") as data_file:
            data_file.write(str(prog_time) + ";" + str(prog_datetime) + ";" + str(data) + ";0.56mA" + ";Channel" + str(commands[1]) + ";" + str(j) + "\n") 

    if len(commands) == 3:
        f = 1
        while f < 101:
            f = f+1
            command = bridgecommands2[0]

            #send command to bridge: 
            ser.write(command.encode("ascii"))

            data = str()
            while(not data):
                data = ser.readline().decode("utf-8").strip()

            #print(data)

            ## append serial data:
            prog_datetime = datetime.now()
            prog_time = time.time() - start_time
            prog_time = prog_time/60

            #print(str(prog_time) + ";" + str(data) + ";1mA" + ";Channel" + str(commands[1]) + ";" + str(f) + "\n") 

            # save data to .txt here:
            with open(file_name, "a") as data_file:
                data_file.write(str(prog_time) + ";" + str(prog_datetime) + ";" + str(data) + ";0.56mA" + ";Channel" + str(commands[2]) + ";" + str(f) + "\n") 


        j = 1
        while j < 101:
            j = j+1

            command = bridgecommands2[1]
            #print(command)

            #send command to bridge: 
            ser.write(command.encode("ascii"))

            data = str()
            while(not data):
                data = ser.readline().decode("utf-8").strip()

            #print(data)


            ## append serial data:
            prog_datetime = datetime.now()
            prog_time = time.time() - start_time
            prog_time = prog_time/60


            # save data to .txt here:
            with open(file_name, "a") as data_file:
                data_file.write(str(prog_time) + ";" + str(prog_datetime) + ";" + str(data) + ";0.56mA" + ";Channel" + str(commands[2]) + ";" + str(j) + "\n") 

"""
if data:
    print("data was successfully retreived")
else:
    print(timeout_message)
"""